# Machine Learning Exercise: Heart Disease from PostgreSQL

This worked solution keeps the database workflow: inspect and join five tables,
correct the target without changing the source, and compare classifiers.

Run cells in order after configuring `.env`. All database access is read-only.
The model classifies recorded disease in historical teaching data, not future
heart attacks. Complete the exercise first, then compare your choices.

## Step 1 - Define the Goal and Research Question

**Question:** can the 13 recorded diagnostic benchmark attributes distinguish
records with disease from records without disease?

1. Join the five tables in the `heart` schema, using patient IDs as keys.
2. Check join coverage, duplicate profiles, category codes, and missing values.
3. Derive `disease_present = 1 - heart_attack`; class 1 means disease present.
4. Split before EDA, then fit preprocessing and logistic regression in a pipeline.
5. Tune on training folds, compare candidates on validation data, and evaluate
   the frozen approach once on test data.

We assume the selected measurements already exist at prediction time.
That assumption would not hold for a tool used before the diagnostic tests.

### Success Metric

For this exercise, select models by **F2 for disease present (`pos_label=1`)**,
and also report precision, recall, and accuracy. This is a teaching choice
that emphasizes missed disease records; it is not a clinical requirement.

| Outcome | Interpretation |
|---|---|
| True positive (TP) | Disease present, predicted present |
| False negative (FN) | Disease present, predicted absent |
| False positive (FP) | No disease recorded, predicted present |
| True negative (TN) | No disease recorded, predicted absent |

- **Recall** = TP / (TP + FN): how many recorded disease cases are found?
- **Precision** = TP / (TP + FP): how many positive predictions are correct?
- **Accuracy** = (TP + TN) / all records: how many predictions are correct overall?

$$
F_2 = \frac{5\,TP}{5\,TP + FP + 4\,FN}
$$

F2 emphasizes recall more than F1. The coefficient 4 on FN comes from
`beta=2`; it does **not** establish that a missed case costs exactly four
times a false alarm in practice. A classifier that calls everyone positive
has perfect recall but many false positives, so a high recall alone is not enough.
Use the same metric and class definition throughout the comparisons.

See [scikit-learn's F-beta definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

### What to Use and Keep in Mind

- **Scale your data:** compare unscaled, standardized, and min-max-scaled pipelines.
- **Tune your model:** use modest randomized and grid searches with the same CV folds.
- **Keep the metric consistent:** score disease present as class 1 throughout.

### Data Overview

This is a **teaching adaptation of the UCI Cleveland Heart Disease dataset**,
split across five PostgreSQL tables. One joined row represents a patient record.
The task is retrospective classification of **recorded disease**, not predicting
a future heart attack. The data is small and historical; this is not a medical
decision tool. [Original UCI documentation](https://archive.ics.uci.edu/dataset/45/heart+disease).

#### Target: Do Not Trust the Column Name

The database's `heart_attack` field has **reversed binary coding** compared
with the original UCI disease-present label:

| Database `heart_attack` | Derived `disease_present` | Meaning |
|---|---|---|
| 1 | 0 | No disease recorded under the source definition |
| 0 | 1 | Disease present |

Preserve the original field, check that it contains only 0 and 1, and derive
`disease_present = 1 - heart_attack`. Use **only `disease_present` as the
modelling target**. Class 0 does not mean generally healthy.
Neither target column belongs in the input features.

This mapping was checked against UCI on 2026-10-08: all 303 records with the
required joined tables agree after accounting for category recoding and missing
measurements. A higher heart rate or less angina in database class 1 therefore
does not indicate a reversed clinical relationship: that class means
**no disease recorded**. Use the documented coding, not correlations, to
interpret labels.

#### Feature Dictionary: Database Codes, Not Original UCI Codes

An ECG records the heart's electrical activity. Angina refers to chest pain
associated with reduced blood supply to the heart; the ST segment is part of
the ECG. Numerical codes below are category identifiers, not measurements.

| Database column | Meaning and coding | Treatment |
|---|---|---|
| `patient_id` | Patient key from the join | Check uniqueness; exclude from inputs |
| `age` | Age in years | Numerical |
| `sex` | Recorded sex: 0 = female, 1 = male | Categorical |
| `chest_pain_type` | 0 = asymptomatic, 1 = atypical angina, 2 = non-anginal pain, 3 = typical angina | Categorical |
| `resting_blood_pressure` | Resting blood pressure on admission, mm Hg | Numerical |
| `serum_cholestoral` | Serum cholesterol, mg/dL; spelling follows the schema | Numerical |
| `fasting_blood_sugar` | Above 120 mg/dL: 0 = false, 1 = true | Categorical |
| `resting_electrocardiographic_results` | 0 = probable/definite left ventricular hypertrophy, 1 = normal, 2 = ST-T abnormality | Categorical |
| `maximum_heartrate_achieved` | Maximum exercise heart rate, beats/minute | Numerical |
| `exercise_induced_angina` | 0 = no, 1 = yes | Categorical |
| `oldpeak` | ST depression induced by exercise relative to rest | Numerical |
| `slope_of_the_peak_exercise_st_segment` | 0 = downsloping, 1 = flat, 2 = upsloping | Categorical |
| `number_of_major_vessels_colored_by_flourosopy` | Vessel count 0 to 3; code 4 represents missing | Numerical count; replace 4 with missing |
| `thal` | Test result: 1 = fixed defect, 2 = normal, 3 = reversible defect; code 0 represents missing | Categorical; replace 0 with missing |
| `real_data` | Teaching source tag, sometimes absent | Quality check only; exclude from inputs |
| `heart_attack` | Original, inverted binary field | Preserve for checking; exclude from inputs |
| `disease_present` | Derived label: 0 = no disease recorded, 1 = disease present | Target only |

The database codes differ from the original UCI codes; **do not mix the two
dictionaries**. Keep the database categories as they are and one-hot encode
them. Convert the two missing-value sentinels before imputation.

Why these 13 inputs? They retain the source's diagnostic benchmark attributes.
Vessel counts and `thal` depend on specialized tests, and several other features
come from an exercise test. They would not all exist at an earlier screening
moment. Choosing inputs requires knowing what is available when a prediction
will be made, not simply using every column in the database.

#### What Is Different from the Original File?

The audit found 303 joined records: 264 match all 13 UCI measurements after
recoding, and 39 match the available measurements but have extra missing
age, sex, or maximum-heart-rate values. One original input-and-label profile
is repeated under two IDs, while another original record is absent.

The former left join also retained 50 extra patients without the required
measurement-table records. Distinguish **an absent joined record** from
**a missing measurement in an existing record**: exclude the former, but retain
the latter for training-fitted imputation. After removing the repeated profile,
the current solution uses 302 records. These counts describe the audited
teaching database, not a guaranteed future database state.

## Step 2 - Get the Data

We import the libraries we need, then pull the heart data from the database.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy.stats import loguniform, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sqlalchemy import URL, create_engine, text

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
pd.options.display.float_format = "{:,.2f}".format
RSEED = 42

### Load from the Database

Inspect the five `heart` tables in DBeaver. The query below is the reference
join, not a CSV replacement. It selects explicit fields and aliases the patient
key as `patient_id`.

We require an existing row in each measurement table with an **inner join**.
This removes patients with no joined measurements, while preserving null
measurements inside existing rows. The explicitly fabricated source tag is
excluded; absent tags alone are retained.

Copy `.env.example` to `.env` and fill in the supplied credentials.
The connection URL is built automatically from the fields; passwords do not
need manual URL encoding. Existing complete `DB_STRING` values are supported.
The transaction is read-only; credentials and private exports stay local.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # Build the URL from the fields in .env; URL.create handles special characters.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Inner joins require the measurement records; individual null values remain.
query_string = """
SELECT p.id AS patient_id, p.age, p.sex,
       cp.chest_pain_type,
       bm.serum_cholestoral, bm.fasting_blood_sugar, bm.thal,
       pva.resting_blood_pressure, pva.resting_electrocardiographic_results,
       pva.maximum_heartrate_achieved, pva.exercise_induced_angina, pva.oldpeak,
       pva.slope_of_the_peak_exercise_st_segment,
       pva.number_of_major_vessels_colored_by_flourosopy, pva.real_data,
       hah.heart_attack
FROM heart.patient AS p
JOIN heart.chest_pain AS cp ON p.id = cp.patient_id
JOIN heart.blood_metrics AS bm ON p.id = bm.patient_id
JOIN heart.pressure_vessels_angina AS pva ON p.id = pva.patient_id
JOIN heart.heart_attack_history AS hah ON p.id = hah.patient_id
WHERE pva.real_data <> 'Evgeny likes white wine for lunch and red wine for dinner'
   OR pva.real_data IS NULL
ORDER BY p.id
"""
# The database is unchanged; close the connection even if reading fails.
try:
    with db.connect() as connection, connection.begin():
        connection.execute(text("SET TRANSACTION READ ONLY"))
        df_sqlalchemy = pd.read_sql(text(query_string), connection)
finally:
    db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optional export: create the ignored data directory only when needed.
# (PROJECT_ROOT / "data").mkdir(exist_ok=True)

In [ ]:
# Optional: save the completed join locally, never to a tracked file.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

In [ ]:
# Continue directly from the query result; no CSV round trip is needed.
df = df_sqlalchemy.copy()

In [ ]:
df.head()

In [ ]:
print(f"Eligible observations: {df.shape[0]}")
print(f"Columns (including identifier, provenance and target): {df.shape[1]}")

### Structural Cleanup Before the Split

Keep the existing short column aliases used below. Check one nonmissing
patient key per joined row; conflicting rows need a join correction, not
silent deduplication.

Validate the raw target, preserve it, and derive `disease_present` so that
class 1 means disease. Convert known missing-value sentinels to `NaN`;
this uses the dictionary, not learned statistics.

Remove exact input-and-label profiles before splitting, ignoring the ID and
source tag. The audit found one repeated UCI profile under two database IDs;
removing that repeated profile prevents exact-observation overlap across splits.
It does not prove the records describe the same person. Ordinary missing
measurements remain for training-fitted imputation.

In [ ]:
# Renaming the column names

df.rename(
    columns={
        "fasting_blood_sugar": "fasting_bs",
        "resting_blood_pressure": "resting__bp",
        "resting_electrocardiographic_results": "electrocardiography",
        "maximum_heartrate_achieved": "max_heartrate",
        "exercise_induced_angina": "exercise_ind_angina",
        "slope_of_the_peak_exercise_st_segment": "slope_of_peak_exercise",
        "number_of_major_vessels_colored_by_flourosopy": "major_vessels_colored",
    },
    inplace=True,
)

In [ ]:
# Check join integrity before silently removing any records.
if df["patient_id"].isna().any() or df["patient_id"].duplicated().any():
    raise ValueError("Expected one nonmissing patient_id per row. Fix the join first.")
if not df["heart_attack"].dropna().isin([0, 1]).all():
    raise ValueError(
        "The raw database target must contain only the documented codes 0 and 1."
    )
print(f"Missing target labels: {int(df['heart_attack'].isna().sum())}")

# Database category codes are different from the original UCI codes.
allowed_codes = {
    "sex": {0, 1},
    "chest_pain_type": {0, 1, 2, 3},
    "fasting_bs": {0, 1},
    "electrocardiography": {0, 1, 2},
    "exercise_ind_angina": {0, 1},
    "slope_of_peak_exercise": {0, 1, 2},
    "major_vessels_colored": {0, 1, 2, 3, 4},
    "thal": {0, 1, 2, 3},
}
for column, allowed in allowed_codes.items():
    unexpected = set(df[column].dropna().unique()) - allowed
    if unexpected:
        raise ValueError(f"Unexpected codes in {column}: {unexpected}")

In [ ]:
# Unknown labels cannot be used for supervised learning; report exclusions.
before = len(df)
df = df.dropna(subset=["heart_attack"]).copy()
print(f"Excluded {before - len(df)} records without a target.")

# Preserve the raw label; correct its meaning in a new modelling target.
df["heart_attack"] = df["heart_attack"].astype(int)
df["disease_present"] = 1 - df["heart_attack"]

# Decode missing sentinels without learning replacement values from the data.
df["major_vessels_colored"] = df["major_vessels_colored"].replace(4, np.nan)
df["thal"] = df["thal"].replace(0, np.nan)

# Ignore IDs and source tags when checking identical input-and-label profiles.
profile_cols = [
    column
    for column in df.columns
    if column not in ("patient_id", "real_data", "heart_attack")
]
repeated_profiles = df.duplicated(subset=profile_cols)
print(f"Repeated input-and-label profiles removed: {int(repeated_profiles.sum())}")
df = df.loc[~repeated_profiles].copy()
if set(df["disease_present"].unique()) != {0, 1}:
    raise ValueError(
        "This exercise requires both disease-present and disease-absent records."
    )
print(f"Retained {len(df)} records, including individual missing measurements.")
pd.crosstab(df["heart_attack"], df["disease_present"])

## Step 3 - Train / Validation / Test Split

Reserve 20% for the final test. Split the remainder into training and validation (60% / 20% of the full sample). Stratification preserves approximate class proportions. This assumes independent patients from the same source population.

EDA and CV searches use training data only. The validation set compares candidates; the test set remains untouched until Step 9.

The second split takes 25% of the remaining 80%, yielding approximately 60% training, 20% validation, and 20% test overall.

<img src="../assets/heart-evaluation-protocol.svg" width="650" alt="Heart exercise: training with internal cross-validation, separate validation for candidate selection, refit on development data, and one final test evaluation.">

Unlike the Titanic notebook, this solution uses a separate validation set after training-only cross-validation. Both approaches are valid; the shared rule is that test data never selects the model. Percentages below describe this exercise, not a universal requirement.

In [ ]:
development_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RSEED, stratify=df["disease_present"]
)
train_df, val_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=RSEED,
    stratify=development_df["disease_present"],
)
print("Training / validation / test:", len(train_df), len(val_df), len(test_df))

## Step 4 - Data Exploration

All exploration below runs on `train_df`. If you split your features and target separately instead, you can rebuild a single frame for EDA with `pd.concat([X_train, y_train], axis=1)`.

### Understand the Basics

In [ ]:
# Type and non-null counts
train_df.info()

In [ ]:
# Descriptive statistics
train_df.describe().T

In [ ]:
# Unique values per column
train_df.nunique()

### Check for Missing Values

In [ ]:
# Share of missing values per column
train_df.isnull().mean()

In [ ]:
# Class balance of the target
train_df["disease_present"].value_counts()

### Visualize the Data

In [ ]:
# Plot all predictive features and the target, excluding identifiers and provenance.
features = [
    c for c in train_df.columns if c not in ("patient_id", "real_data", "heart_attack")
]
nrows = (len(features) + 2) // 3
fig, axes = plt.subplots(nrows, 3, figsize=(15, 3 * nrows), squeeze=False)
for item, ax in zip(features, axes.flat):
    # Counts remain meaningful for category codes; a density curve would imply continuity.
    sns.histplot(train_df[item], ax=ax, color="#33658A").set(title=item, xlabel="")
for ax in list(axes.flat)[len(features) :]:
    ax.set_visible(False)
fig.tight_layout()

Read the class counts above and compute the proportions. Do not assume the live database is balanced or has the same distribution as an earlier run.

### Identify Outliers

In [ ]:
# Inspect outliers in continuous measurements, not arbitrary category codes.
continuous_features = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
]
train_df[continuous_features].plot(
    kind="box", subplots=True, layout=(2, 3), figsize=(12, 7)
)
plt.tight_layout()
plt.show()

### Missing Measurements and Source Tags

The database adds missing age, sex, and maximum-heart-rate values compared
with the source. Treat these as missing measurements, not new categories.
Keep the original `real_data` tags and use a plotting copy to show absent tags.

Tags were not sufficient to establish source identity: the UCI comparison
checked the available measurements and target instead. Similar-looking
distributions alone do not prove shared provenance.

In [ ]:
# Inspect missing features without overwriting the original provenance tags.
print(train_df[continuous_features].isna().mean().rename("missing_fraction"))
provenance_plot = train_df.copy()
provenance_plot["source_label"] = train_df["real_data"].fillna("Missing source tag")

In [ ]:
# Compare source groups using measurements, not IDs or arbitrary categorical codes.
sns.pairplot(
    provenance_plot, vars=continuous_features, hue="source_label", corner=True, height=2
);

Absent source tags are not proof that a record is fabricated.
The source comparison found that all 303 fully joined records agree with
an original UCI profile on their available measurements, after recoding.
The repeated profile is removed before splitting.
Imputers will handle the remaining missing inputs using training data only.

Inspect maximum exercise heart rate and other measurements by the corrected
`disease_present` classes: 0 = no disease recorded, 1 = disease present.
The old raw database class 1 meant no disease; do not reuse that interpretation
for the corrected target. These plots are exploratory, not causal evidence.

In [ ]:
# Compare the measurements using the corrected class meanings.
class_summary = train_df.groupby("disease_present").agg(
    records=("patient_id", "size"),
    mean_max_heart_rate=("max_heartrate", "mean"),
    exercise_angina_pct=("exercise_ind_angina", lambda values: 100 * values.mean()),
)
class_summary.rename(index={0: "No disease recorded", 1: "Disease present"})

In [ ]:
# Keep the graphical comparison by target class as well as the numeric summary.
sns.pairplot(
    train_df, vars=continuous_features, hue="disease_present", corner=True, height=2
)
plt.show()
train_df.groupby("disease_present")[continuous_features].median()

Describe the class relationships actually visible in this training sample.
They do not establish future heart-attack risk, and a single measurement
does not determine the diagnosis.

### Correlation Analysis

In [ ]:
# Correlation heatmap

fig, ax = plt.subplots(figsize=(12, 8))

# Select measurements and the binary target; exclude IDs and category codes
df_numeric = train_df[continuous_features + ["disease_present"]]

# Compute correlations
correlations = df_numeric.corr()

# Generate a mask for the upper triangle
mask = np.zeros_like(correlations)
mask[np.triu_indices_from(mask)] = True

# Generate a custom diverging colormap
cmap = sns.diverging_palette(140, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio

sns.heatmap(
    correlations,
    mask=mask,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    linewidths=1,
    cbar_kws={"shrink": 0.7},
    ax=ax,
);

This heatmap shows Pearson associations in training data, not causal effects.
The corrected binary target can be included, but arbitrary category codes,
IDs, source tags, and the raw inverted label are not continuous predictors.

## Step 5 - Feature Engineering

Retain individual missing inputs for imputation inside the pipeline.
Exclude `patient_id`, `real_data`, **the raw `heart_attack` field**, and the
derived `disease_present` target from input features. The raw field directly
reveals the answer after inversion and would cause target leakage.

Median imputation handles numerical inputs; categorical inputs use their
most frequent training category. The known sentinel codes were already decoded,
but replacement values are learned only inside each training fold.
`keep_empty_features=True` preserves an entirely missing column in a fold;
investigate such a case rather than treating its fallback as a measurement.

We keep all 13 diagnostic benchmark inputs. Specialized-test measurements
such as vessel count and `thal` would not all be available for early screening.

In [ ]:
# Retain missing inputs: the pipeline below will learn replacement values.
print("Training / validation / test rows:", len(train_df), len(val_df), len(test_df))

### Define Features and Target

In [ ]:
# Group columns by treatment, excluding identifiers and provenance from predictions.
num_cols = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
    "major_vessels_colored",
]
cat_cols = [
    "sex",
    "fasting_bs",
    "exercise_ind_angina",
    "chest_pain_type",
    "thal",
    "electrocardiography",
    "slope_of_peak_exercise",
]
feature_cols = num_cols + cat_cols
assert "heart_attack" not in feature_cols and "disease_present" not in feature_cols

X_train, y_train = train_df[feature_cols], train_df["disease_present"]
X_val, y_val = val_df[feature_cols], val_df["disease_present"]

# Only for the deliberately flawed identifier demonstration.
X_train_id = train_df[feature_cols + ["patient_id"]]
X_val_id = val_df[feature_cols + ["patient_id"]]
print("Training / validation shapes:", X_train.shape, X_val.shape)

### Build the Feature-Engineering Pipeline

Following the first notebook, all feature engineering goes into a single `ColumnTransformer` that imputes and scales the numeric columns and imputes and one-hot encodes the categorical ones. We wrap it with the model in a `Pipeline`, so each transformer is fit on the training folds only, even inside cross-validation. The helper below builds that pipeline for a given scaler.

In [ ]:
def build_pipeline(scaler="passthrough", numeric=None):
    if numeric is None:
        numeric = num_cols
    # Fit imputation before scaling within each training fold.
    numerical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", scaler),
        ]
    )
    # Fill missing categories before encoding; learn both steps on training folds.
    categorical = Pipeline(
        [
            (
                "impute",
                SimpleImputer(strategy="most_frequent", keep_empty_features=True),
            ),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    features = ColumnTransformer(
        [
            ("num", numerical, numeric),
            ("cat", categorical, cat_cols),
        ]
    )
    return Pipeline(
        [
            ("features", features),
            ("model", LogisticRegression(max_iter=10000, random_state=RSEED)),
        ]
    )

All one-hot indicators are retained for this regularized comparison.
With `handle_unknown="ignore"`, an unseen category becomes an all-zero block;
it is not confused with an intentionally dropped reference category.
Such inputs still deserve investigation in a real system.

## Steps 6 to 8 - Model Training and Hyperparameter Tuning

Feature engineering, model training, hyperparameter tuning, and validation form an iterative loop. We start from a simple baseline, then test whether scaling and tuned hyperparameters improve on it. Improvement is not guaranteed.

In [ ]:
# All comparisons score disease present as class 1.
validation_results = {}
class_names = ["No disease recorded", "Disease present"]


def evaluate(name, y_true, y_pred):
    metrics = {
        "F2": fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
    }
    print(f"{name}: accuracy {metrics['Accuracy']:.2f} | F2 {metrics['F2']:.2f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        confusion_matrix(y_true, y_pred, labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="YlGnBu",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set(title=name, xlabel="Predicted status", ylabel="Recorded status")
    fig.tight_layout()
    plt.show()
    return metrics

### Baseline Models

Keep the original simple `max_heartrate` rule: predict the class whose training mean is nearer. The boundary is the midpoint of the two class means; its direction is learned from the training labels rather than assuming a higher heart rate means higher risk. Missing measurements use the training median. This is a teaching heuristic, not a clinical rule.

Also compare a constant-positive classifier: recall is 1 when positives are present, but every negative becomes a false positive. This extra reference shows why a high F2 alone is not enough.

In [ ]:
def predict_heartrate_rule(training, rows):
    """Learn a midpoint rule from training data and apply it to new rows."""
    observed = training["max_heartrate"].dropna()
    if observed.empty:
        raise ValueError("The heart-rate rule needs observed training measurements.")
    # Learn replacements and the class boundary from training observations only.
    fill_value = observed.median()
    class_means = (
        training["max_heartrate"]
        .fillna(fill_value)
        .groupby(training["disease_present"])
        .mean()
    )
    print("Training class mean heart rates:", class_means.round(1).to_dict())
    print("Midpoint threshold:", round(class_means.mean(), 1))
    values = rows["max_heartrate"].fillna(fill_value)
    # Nearest class mean; exact ties predict class 0.
    return ((values - class_means[1]).abs() < (values - class_means[0]).abs()).astype(
        int
    )


validation_results["Heart-rate rule"] = evaluate(
    "Heart-rate rule (validation)", y_val, predict_heartrate_rule(train_df, X_val)
)

baseline = DummyClassifier(strategy="constant", constant=1)
baseline.fit(X_train, y_train)
validation_results["Constant positive baseline"] = evaluate(
    "Baseline (validation)", y_val, baseline.predict(X_val)
)

### First Logistic Regression (and a Leakage Trap)

Now a real model. We fit a logistic regression through the pipeline, first without `patient_id` and then with it, to show how an identifier can leak the target.

In [ ]:
# Leave numeric features unscaled; impute and encode within the pipeline.
log_reg = build_pipeline()
log_reg.fit(X_train, y_train)
validation_results["Unscaled logistic regression"] = evaluate(
    "Unscaled logistic regression (validation)", y_val, log_reg.predict(X_val)
)

In [ ]:
# Deliberately include the identifier to illustrate an invalid candidate.
log_reg_id = build_pipeline(numeric=num_cols + ["patient_id"])
log_reg_id.fit(X_train_id, y_train)
evaluate(
    "Identifier included: invalid candidate (validation)",
    y_val,
    log_reg_id.predict(X_val_id),
)

If adding `patient_id` improves performance sharply, suspect a collection or ordering artifact. An ID is not automatically leakage, but a label-correlated identifier without predictive meaning is not a defensible feature. This deliberately flawed model is excluded from selection.

In [ ]:
sns.kdeplot(data=train_df, x="patient_id", hue="disease_present", fill=True);

Inspect whether labels occupy different ID ranges. The pattern can change when the teaching database changes, so do not assume a fixed cutoff.

In [ ]:
_ = sns.scatterplot(data=train_df, x="patient_id", y="disease_present")

In [ ]:
# Check how many training observations belong to each class.
train_df.groupby("disease_present")["patient_id"].count().rename("count")

In [ ]:
# Compare class-specific ID ranges without assuming a fixed database cutoff.
id_ranges = train_df.groupby("disease_present")["patient_id"].agg(["min", "max"])
id_ranges

In [ ]:
# Overlapping ranges do not rule out ordering artifacts; use the plots too.
overlap_start = id_ranges["min"].max()
overlap_end = id_ranges["max"].min()
print("Class ID ranges overlap:", overlap_start <= overlap_end)

Database ordering can encode how records were imported or sorted. A random row split may preserve such artifacts in every subset and yield misleadingly high scores. No fixed patient-ID threshold is part of this solution.

### Training with Feature Scaling

Compare standardization with min-max scaling using the same validation set. Only the scaler changes; preprocessing and model choices otherwise remain fixed.

#### Standardization

In [ ]:
std_pipe = build_pipeline(StandardScaler())
std_pipe.fit(X_train, y_train)
validation_results["Standardized logistic regression"] = evaluate(
    "Standardized (validation)", y_val, std_pipe.predict(X_val)
)

#### Min-Max Scaling

In [ ]:
minmax_pipe = build_pipeline(MinMaxScaler())
minmax_pipe.fit(X_train, y_train)
validation_results["Min-max logistic regression"] = evaluate(
    "Min-max (validation)", y_val, minmax_pipe.predict(X_val)
)

### Hyperparameter Tuning with RandomizedSearchCV

Search the full standardized pipeline using stratified training folds. Keep the budget modest: ten sampled combinations and five folds. Vary `C` (smaller means stronger regularization) and `l1_ratio` (0 = L2, 1 = L1, intermediate values = elastic net). Use `saga`, which supports all three.

L1 can set coefficients to zero; L2 shrinks coefficients without generally setting them exactly to zero. The random search samples mixtures; the grid below explicitly compares the two endpoints.

[Reference: scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

The `model__` prefix addresses parameters of the pipeline's `model` step; preprocessing remains inside each fitted fold.

In [ ]:
# F2 gives recall more weight; inspect precision and error counts as well.
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
if y_train.value_counts().min() < cv.n_splits:
    raise ValueError("Need at least five training examples per class for five-fold CV.")

random_search = RandomizedSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": loguniform(0.01, 10),
        "model__l1_ratio": uniform(0, 1),
    },
    n_iter=10,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RSEED,
    verbose=1,
    error_score="raise",
)
random_search.fit(X_train, y_train)
print("Best parameters:", random_search.best_params_)
print(f"Best training CV F2: {random_search.best_score_:.3f}")

### Hyperparameter Tuning with GridSearchCV

Evaluate four C values for each of L2 (`l1_ratio=0`) and L1 (`l1_ratio=1`): eight combinations, using the same folds and metric as the random search. Compare **CV** results to select the tuning method before using the validation set for the final candidate comparison. Neither search guarantees the best possible model outside its search space.

These C values are evenly spaced on a logarithmic scale, equivalent to `np.logspace(-2, 1, num=4)`.

In [ ]:
# Compare L2 and L1 with the same folds and metric as randomized search.
grid_search = GridSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__l1_ratio": [0, 1],
    },
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    error_score="raise",
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print(f"Best training CV F2: {grid_search.best_score_:.3f}")

selected_search = max(
    [grid_search, random_search], key=lambda search: search.best_score_
)
tuned_model = selected_search.best_estimator_
validation_results["Tuned logistic regression"] = evaluate(
    "Tuned model (validation)", y_val, tuned_model.predict(X_val)
)
# Compare eligible candidates on the same validation rows; keep test results separate.
validation_comparison = pd.DataFrame.from_dict(validation_results, orient="index")
validation_comparison.sort_values("F2", ascending=False, kind="stable")

## Step 9 - Calculate Test Score

Choose the eligible candidate with the highest validation F2, including both baselines. Ties follow their insertion order in `validation_results`. Refit the selected model (or relearn the heart-rate rule) on training plus validation data, then evaluate **once** on the test set.

Search objects already refit on their training input, but they have not seen the separate validation subset. Report the final confusion matrix, precision, recall, and F2. If the baseline wins, report that outcome rather than claiming the more complex model is better.

Read precision and recall alongside F2: the constant-positive baseline has recall 1 but its precision equals the positive-class proportion. F2 favors recall, so a high F2 alone does not imply a useful model. Metrics refer to class 1; undefined ratios are reported as 0 by convention. The table is rounded for display only; selection uses full-precision F2.

In [ ]:
candidates = {
    "Constant positive baseline": baseline,
    "Unscaled logistic regression": log_reg,
    "Standardized logistic regression": std_pipe,
    "Min-max logistic regression": minmax_pipe,
    "Tuned logistic regression": tuned_model,
}
selected_name = max(validation_results, key=lambda name: validation_results[name]["F2"])
X_test, y_test = test_df[feature_cols], test_df["disease_present"]
if selected_name == "Heart-rate rule":
    y_pred = predict_heartrate_rule(development_df, X_test)
else:
    # Refit the selected configuration on development data before the final test.
    best_model = clone(candidates[selected_name])
    best_model.fit(development_df[feature_cols], development_df["disease_present"])
    y_pred = best_model.predict(X_test)
print(f"Selected using validation data: {selected_name}")
test_metrics = evaluate("Final test", y_test, y_pred)
print(
    classification_report(
        y_test, y_pred, labels=[0, 1], target_names=class_names, zero_division=0
    )
)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
print(f"Detected disease records: {tp}; missed disease records: {fn}.")
print(f"False positives: {fp}; correctly negative records: {tn}.")

## Step 10 - Deployment and Limitations

The database keeps a useful SQL and data-quality exercise, but this model
must **not** be deployed as a medical decision tool.
The source is a small historical diagnostic sample, not a representative
population. The teaching adaptation adds missing measurements, changes codes,
and contains a repeated profile; its corrected target is recorded disease,
not a future event.

One small random test split does not establish clinical validity, calibrated
probabilities, fairness, or safety. Do not retune after inspecting test errors.

For a real project, define the prediction moment and which measurements exist
then. Specialized tests may be unavailable or too costly.
Obtain appropriate domain review and independent representative validation.
Monitor input quality, distribution shift, subgroup performance, and error
costs, not just overall accuracy.

## Check Your Understanding - Answers

1. **Why exclude `patient_id`?** It identifies a patient rather than a generalizable measurement. It may encode collection order or source and produce a misleading shortcut. Keep it for joins and duplicate checks, not as a predictor.
2. **Why fit imputation inside each fold?** Medians or other replacement values must come only from that fold's training rows. Computing them before cross-validation lets held-out rows influence preprocessing and can bias evaluation. The pipeline enforces the boundary.
3. **Why not select using test scores?** Choosing the best test score uses the test set for model selection, making it an optimistic estimate of the selected model's performance. Freeze choices using training CV and validation, refit on development data, then evaluate once on test.
4. **Why exclude the raw `heart_attack` column too?** It directly determines
   `disease_present = 1 - heart_attack`. Keeping it as an input would give the
   classifier the answer. Preserve it only for auditing the target mapping.